In [1]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LinearRegression 
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import mean_squared_error, r2_score 
import matplotlib.pyplot as plt 
# Create realistic house price 
# dataset 
np.random.seed(42) 
n_samples = 1000 
# Features: size, bedrooms, age, location_score 
size = np.random.normal(2000, 500, n_samples) 
bedrooms = np.random.poisson(3, n_samples) 
age = np.random.exponential(10, n_samples) 
location_score = np.random.uniform(1, 10, n_samples) 
# Target: price (with realistic relationship) 
price = (size * 150 + bedrooms * 10000 + (20 - age) * 1000 + location_score * 5000 + np.random.normal(0, 20000, n_samples)) 
# Create DataFrame 
df = pd.DataFrame({ 'size': size, 'bedrooms': bedrooms, 'age': age, 'location_score': location_score, 'price': price }) 
print("Dataset created:") 
print(df.head()) 
print(f"\nDataset shape: {df.shape}") 
print(f"Price range: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")

Dataset created:
          size  bedrooms        age  location_score          price
0  2248.357077         1   2.140307        4.150062  388580.752085
1  1930.867849         2   1.381951        9.355353  373974.889836
2  2323.844269         1   3.536216        1.590635  380096.027008
3  2761.514928         7  15.143463        5.944706  519668.125751
4  1882.923313         6   6.379516        6.317796  408104.792321

Dataset shape: (1000, 5)
Price range: $124,653 - $627,023


In [19]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Sample DataFrame (you can replace this with your actual housing dataset)
# df = pd.read_csv('your_data.csv')  # Uncomment if loading from file
df = pd.DataFrame({
    'price': [200000, 300000, 250000, 400000, 180000],
    'total_rooms': [1000, 1500, 1200, 1800, 900],
    'households': [2, 3, 2.5, 4, 1.8],
    'population': [500, 700, 600, 1000, 400],
    'housing_median_age': [10, 20, 15, 30, 25],
    'median_income': [2.5, 5.5, 7.5, 10.5, 3.5],
    'ocean_proximity': ['NEAR BAY', 'INLAND', 'NEAR OCEAN', 'INLAND', 'ISLAND'],
    'latitude': [37.0, 36.5, 35.9, 36.8, 37.2],
    'longitude': [-122.0, -121.9, -122.3, -121.8, -122.1]
})

# Preprocess
df_processed = df.copy()

# Feature Engineering Function
def create_features(df):
    df_features = df.copy()

    df_features['price_per_sqft'] = df_features['price'] / df_features['total_rooms']
    df_features['rooms_per_household'] = df_features['total_rooms'] / df_features['households']
    df_features['population_density'] = df_features['population'] / df_features['households']
    df_features['house_age'] = 2024 - df_features['housing_median_age']
    df_features['income_category'] = pd.cut(
        df_features['median_income'],
        bins=[0, 3, 6, 10, np.inf],
        labels=['Low', 'Medium', 'High', 'Very High']
    )
    
    if 'ocean_proximity' in df_features.columns:
        dummies = pd.get_dummies(df_features['ocean_proximity'], prefix='ocean')
        df_features = pd.concat([df_features, dummies], axis=1)
        df_features.drop('ocean_proximity', axis=1, inplace=True)

    if 'latitude' in df_features.columns and 'longitude' in df_features.columns:
        coords = df_features[['latitude', 'longitude']]
        kmeans = KMeans(n_clusters=3, random_state=42)  # Reduced clusters for small sample
        df_features['location_cluster'] = kmeans.fit_predict(coords)

    return df_features

# Apply Feature Engineering
housing_engineered = create_features(df_processed)

# Summary
print(f"Features after engineering: {housing_engineered.shape[1]}")
print("New features created:")
for f in ['price_per_sqft', 'rooms_per_household', 'population_density', 
          'house_age', 'income_category', 'location_cluster']:
    if f in housing_engineered.columns:
        print(f"{f}")


Features after engineering: 18
New features created:
price_per_sqft
rooms_per_household
population_density
house_age
income_category
location_cluster


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Prepare data
X = df[['size', 'bedrooms', 'age', 'location_score']]
y = df['price']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate models
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results[name] = {'MSE': mse, 'RMSE': rmse, 'R²': r2}

    print(f"\n{name} Results:")
    print(f"MSE: ${mse:,.0f}")
    print(f"RMSE: ${rmse:,.0f}")
    print(f"R²: {r2:.4f} ({r2*100:.1f}% variance explained)")



Linear Regression Results:
MSE: $449,316,435
RMSE: $21,197
R²: 0.9317 (93.2% variance explained)

Random Forest Results:
MSE: $584,333,859
RMSE: $24,173
R²: 0.9112 (91.1% variance explained)


In [ ]:
# Common features in housing datasets
features = {
    'property_features': ['bedrooms', 'bathrooms', 'square_feet', 'lot_size', 'year_built'],
    'location_features': ['zip_code', 'latitude', 'longitude', 'school_district'],
    'neighborhood_features': ['crime_rate', 'avg_income', 'walkability_score'],
    'market_features': ['days_on_market', 'price_per_sqft', 'seasonal_trends'],
    'amenities': ['garage', 'pool', 'fireplace', 'air_conditioning']
}


import pandas as pd
from sklearn.datasets import fetch_california_housing
import requests

# Load California Housing Dataset
def load_housing_data():
    housing = fetch_california_housing()
    df = pd.DataFrame(housing.data, columns=housing.feature_names)
    df['price'] = housing.target
    return df

# Alternative: Load from CSV
def load_custom_data(file_path):
    df = pd.read_csv(file_path)
    return df

# Load data
housing_data = load_housing_data()
print(f"Dataset shape: {housing_data.shape}")
print(housing_data.head())

Dataset shape: (20640, 9)
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  price  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  
